# 01 — Initial Data Understanding & EDA

**Scope:** City Hotel H2 only. This notebook describes the raw data; it does not modify the CSV, drop or clean rows, or fit regression, classification, or clustering models. All plots and tables are descriptive. Run from the repository root or from this notebook's directory.

The course brief requires inspecting data dimensions/types, duplicates, missingness, invalid or unusual values, category cardinality, ranges, descriptive statistics, skew/outliers, and leakage risks. This notebook covers that initial understanding for Member 1.


## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42  # Reserved for any reproducible sampling added to exploratory work.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', context='notebook')


## 2. Load the required dataset

In [ ]:
# Resolve the path when launched either at the project root or in notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'H2.csv').is_file():
    PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'H2.csv'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Expected the project dataset at {DATA_PATH}')

OUTPUT_FIGURES = PROJECT_ROOT / 'outputs' / 'figures'
OUTPUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {DATA_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')


In [ ]:
df.head()

In [ ]:
print('Column names:')
print(df.columns.tolist())
print('\nData types:')
display(df.dtypes.rename('dtype').to_frame())
print('\nDataFrame info:')
df.info()

## 3. Data quality and schema checks

These checks report observations without changing the dataframe.

In [ ]:
missing = df.isna().sum().rename('missing_count').to_frame()
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(3)
display(missing.loc[missing['missing_count'].gt(0)].sort_values('missing_count', ascending=False))
print(f'Exact duplicate rows: {df.duplicated().sum():,} ({df.duplicated().mean():.2%})')

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
cardinality = pd.DataFrame({
    'unique_non_null': df[categorical_cols].nunique(dropna=True),
    'unique_including_missing': df[categorical_cols].nunique(dropna=False),
    'missing': df[categorical_cols].isna().sum(),
}).sort_values('unique_non_null', ascending=False)
display(cardinality)
print('Observed categorical values (low-cardinality columns):')
for column in cardinality.index:
    if cardinality.loc[column, 'unique_non_null'] <= 15:
        print(f'\n{column}: {df[column].value_counts(dropna=False).to_dict()}')

In [ ]:
# Types worth reviewing: source labels say these are temporal/categorical, identifiers,
# or dates although the CSV parser represents them as strings.
suspicious_types = pd.DataFrame([
    ('ArrivalDateMonth', str(df['ArrivalDateMonth'].dtype), 'Temporal category; month order is not alphabetical.'),
    ('ArrivalDateYear / ArrivalDateWeekNumber / ArrivalDateDayOfMonth',
     str(df['ArrivalDateYear'].dtype), 'Integer-valued calendar components; combine/validate as dates before temporal use.'),
    ('ReservationStatusDate', str(df['ReservationStatusDate'].dtype), 'Date-like string; parse and inspect its lifecycle meaning.'),
    ('Agent / Company', f"{df['Agent'].dtype} / {df['Company'].dtype}", 'Identifier-like categories; string parsing avoids treating IDs as quantities.'),
], columns=['column', 'observed_dtype', 'review'])
display(suspicious_types)

In [ ]:
# Range and logical checks flag unusual records for later review; they do not filter anything.
checks = {
    'IsCanceled outside {0,1}': ~df['IsCanceled'].isin([0, 1]),
    'negative LeadTime': df['LeadTime'] < 0,
    'week number outside 1..53': ~df['ArrivalDateWeekNumber'].between(1, 53),
    'day of month outside 1..31': ~df['ArrivalDateDayOfMonth'].between(1, 31),
    'negative stay nights': (df['StaysInWeekendNights'] < 0) | (df['StaysInWeekNights'] < 0),
    'negative guest count': (df[['Adults', 'Children', 'Babies']] < 0).any(axis=1),
    'no adults, children, or babies': df[['Adults', 'Children', 'Babies']].eq(0).all(axis=1),
    'negative ADR': df['ADR'] < 0,
    'zero ADR': df['ADR'].eq(0),
    'ADR above 99th percentile': df['ADR'] > df['ADR'].quantile(.99),
    'unparseable reservation status date': pd.to_datetime(df['ReservationStatusDate'], errors='coerce').isna(),
}
quality_flags = pd.DataFrame({name: [int(mask.sum()), float(mask.mean() * 100)] for name, mask in checks.items()},
                             index=['rows', 'percent']).T
display(quality_flags)

arrival_month = pd.to_datetime(df['ArrivalDateMonth'], format='%B', errors='coerce').dt.month
arrival_date = pd.to_datetime({
    'year': df['ArrivalDateYear'], 'month': arrival_month,
    'day': df['ArrivalDateDayOfMonth'],
}, errors='coerce')
print('Arrival date components not forming a valid calendar date:', int(arrival_date.isna().sum()))
print('ReservationStatus values:', df['ReservationStatus'].value_counts(dropna=False).to_dict())


## 4. Descriptive statistics

In [ ]:
display(df.describe(include='all').T)
numeric_summary = df.describe().T
numeric_summary.to_csv(OUTPUT_TABLES / 'numeric_descriptive_statistics.csv')

## 5. Important numeric variables

In [ ]:
focus_columns = ['ADR', 'LeadTime', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children']
focus_summary = df[focus_columns].describe(percentiles=[.01, .25, .5, .75, .95, .99]).T
focus_summary['missing'] = df[focus_columns].isna().sum()
display(focus_summary)
focus_summary.to_csv(OUTPUT_TABLES / 'important_numeric_summary.csv')

## 6. Cancellation, market segment, and ADR

In [ ]:
cancellation_rate = df['IsCanceled'].mean()
print(f'Cancellation rate: {cancellation_rate:.2%} ({df["IsCanceled"].sum():,} of {len(df):,} bookings)')
segment_summary = (df.groupby('MarketSegment', dropna=False)['IsCanceled']
                   .agg(bookings='size', cancellation_rate='mean')
                   .sort_values('bookings', ascending=False))
segment_summary['cancellation_rate'] *= 100
display(segment_summary.round(2))
segment_summary.to_csv(OUTPUT_TABLES / 'cancellation_by_market_segment.csv')
print('ADR percentiles:', df['ADR'].quantile([.5, .9, .95, .99, .999]).round(2).to_dict())
print(f'ADR values equal to zero: {df["ADR"].eq(0).sum():,}; max ADR: {df["ADR"].max():,.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=df, x='IsCanceled', ax=axes[0], color='#4C78A8')
axes[0].set(title=f'Booking outcome (cancellation rate {cancellation_rate:.1%})', xlabel='IsCanceled (0 = no, 1 = yes)', ylabel='Bookings')
segment_order = df['MarketSegment'].value_counts().index
sns.countplot(data=df, y='MarketSegment', order=segment_order, ax=axes[1], color='#59A14F')
axes[1].set(title='Bookings by market segment', xlabel='Bookings', ylabel='Market segment')
# Zoom to the 99th percentile so the main body is legible; report the full range above.
adr_limit = df['ADR'].quantile(.99)
sns.histplot(data=df.loc[df['ADR'].le(adr_limit)], x='ADR', bins=45, ax=axes[2], color='#F28E2B')
axes[2].set(title=f'ADR distribution (through 99th percentile = {adr_limit:.0f})', xlabel='ADR', ylabel='Bookings')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / 'initial_distributions.png', dpi=160, bbox_inches='tight')
plt.show()

## 7. Lead time and cancellation

In [ ]:
lead_bins = [-1, 7, 30, 90, 180, 365, np.inf]
lead_labels = ['0–7', '8–30', '31–90', '91–180', '181–365', '366+']
leadtime_cancellation = (df.assign(LeadTimeBand=pd.cut(df['LeadTime'], bins=lead_bins, labels=lead_labels))
                         .groupby('LeadTimeBand', observed=False)['IsCanceled']
                         .agg(bookings='size', cancellation_rate='mean'))
leadtime_cancellation['cancellation_rate'] *= 100
display(leadtime_cancellation.round(2))
leadtime_cancellation.to_csv(OUTPUT_TABLES / 'cancellation_by_lead_time_band.csv')
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=leadtime_cancellation.index.astype(str), y=leadtime_cancellation['cancellation_rate'], ax=ax, color='#E15759')
ax.set(title='Cancellation rate by lead-time band', xlabel='Lead time (days)', ylabel='Bookings canceled (%)')
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / 'cancellation_by_lead_time.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Temporal patterns supported by arrival columns

In [ ]:
# Use arrival year + month (and day for validation) rather than week number alone.
temporal = df.assign(ArrivalDate=arrival_date)
monthly = (temporal.dropna(subset=['ArrivalDate'])
           .assign(ArrivalMonth=temporal['ArrivalDate'].dt.to_period('M').dt.to_timestamp())
           .groupby('ArrivalMonth')
           .agg(bookings=('IsCanceled', 'size'), cancellation_rate=('IsCanceled', 'mean'), mean_adr=('ADR', 'mean')))
monthly['cancellation_rate'] *= 100
display(monthly.round(2))
monthly.to_csv(OUTPUT_TABLES / 'monthly_arrival_summary.csv')
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.index, monthly['cancellation_rate'], marker='o', linewidth=1.8, color='#B07AA1')
ax.set(title='Cancellation rate by month of arrival', xlabel='Arrival month', ylabel='Bookings canceled (%)')
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / 'monthly_cancellation_rate.png', dpi=160, bbox_inches='tight')
plt.show()

## 9. Findings

### Data quality
- The H2 file has **79,330 rows and 31 columns**, matching the course brief. There are **25,902 exact duplicate rows**; duplicates are flagged for later investigation, not removed in this EDA.
- Missingness is limited to `Country` (**24**) and `Children` (**4**). No rows are dropped and no values are imputed here.
- No negative ADR values or negative stay/guest counts are observed. `ADR` includes **1,208 zero values** and reaches **5,400**, far above its 99th percentile (**228**): inspect these observations and the skew before deciding how to handle them.
- `Adults` is zero on some records, and **167 rows have zero adults, children, and babies recorded**; assess these jointly with booking status before deciding whether they are invalid. Weekend nights reach 16, week nights 41, and `Babies` reaches 10; these tails merit review rather than automatic removal.
- `ReservationStatusDate` and `ArrivalDateMonth` are read as strings. Month is a temporal category; the reservation status date is date-like and should be parsed for analysis. `Agent` and `Company` are identifier-like strings, not numeric quantities.

### EDA findings
- Overall cancellation rate is **41.73%**.
- `Online TA` is the largest segment (**38,748 bookings**), followed by `Offline TA/TO` (**16,747**) and `Groups` (**13,975**). Cancellation rates vary substantially: `Groups` **68.86%**, `Online TA` **37.40%**, and `Direct` **17.33%**. The `Undefined` segment has only two rows, so its rate is unstable.
- Cancellation rate increases across the descriptive lead-time bands: **12.20%** at 0–7 days, **39.91%** at 31–90 days, **62.72%** at 181–365 days, and **71.12%** above 365 days. This is an association, not evidence that longer lead time causes cancellation.
- Arrival year/month/day fields support month-of-arrival summaries over the recorded 2015–2017 period. These are descriptive and reflect the available date window; they do not establish a general seasonal effect.

### Potential Preprocessing Requirements
- **Missing values:** decide and document imputation or missing-category handling for `Country` and `Children` after confirming semantics; compare strategies within training pipelines later.
- **Categorical encoding:** encode low-cardinality fields appropriately. `ArrivalDateMonth` has natural calendar order; `Agent`, `Company`, and `Country` have many categories and need deliberate treatment.
- **Scaling:** consider scaling numeric features for distance- or gradient-based algorithms (for example, KNN, SVM, and PCA); tree-based methods generally do not need feature scaling.
- **Outliers/skew:** inspect the long right tails in ADR, lead time, stay lengths, waiting-list days, prior-booking counts, and identifier frequencies. ADR has extreme high values and many zeros. Do not cap, transform, or remove values until justified for a specific task.
- **High cardinality:** `ReservationStatusDate` has 864 distinct values, `Agent` 224, `Company` 208, and `Country` 166 observed values (167 including missing). Check whether each is useful and stable at the prediction point; avoid treating IDs as ordered numbers.
- **Duplicates:** investigate exact duplicate rows and their provenance before deciding whether they represent repeated bookings or accidental duplication.

### Potential Leakage Risks

Leakage depends on the prediction question and when the prediction is made. The CSV does not define one universal prediction timestamp, so feature safety must be audited separately for ADR prediction, cancellation prediction, and clustering.

- `ReservationStatus` directly states the final outcome (`Canceled`, `Check-Out`, or `No-Show`); it is post-outcome information for predicting `IsCanceled` and can make clusters outcome-dependent.
- `ReservationStatusDate` records when the last reservation status was set. It is lifecycle/future information relative to an early booking-time prediction and may reveal cancellation timing.
- `AssignedRoomType` may be assigned or changed after the original booking; compare its availability at the intended prediction point against `ReservedRoomType`.
- `BookingChanges`, `DaysInWaitingList`, `RequiredCarParkingSpaces`, and `TotalOfSpecialRequests` may accrue or change after initial booking. Their inclusion requires a defined prediction point and a check that the value would already be known.
- `ADR` is the regression target. For cancellation classification, whether ADR is known at the prediction point needs explicit justification; never use a target-derived quantity to predict itself.
- Previous cancellation/booking counts are historical by definition, but verify that they were recorded before the current booking's prediction point. For clustering, exclude outcome/future fields if the goal is to discover pre-outcome booking segments.

No preprocessing, row filtering, feature selection, or model fitting is performed in this notebook. These are investigation notes for later task-specific decisions.
